## Llama 2模型

In [6]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "C:/llama2/llama-2-7b-hf",
    return_dict=True,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="cpu",
)

print(model)

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████| 2/2 [01:40<00:00, 50.29s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNo

## 读取hf模型

In [3]:
import zipfile
import pickle
from pathlib import Path
from typing import (IO, Any, Callable, Optional, List, Dict)
from dataclasses import dataclass
import numpy as np

DT_F16 = 'F16'
DT_F32 = 'F32'
DT_BF16 = 'DT_BF16'
DT_I32 = 'DT_I32'

DataType = 'F16' 

@dataclass
class LazyStorageKind:
    data_type: DataType

@dataclass
class LazyStorage:
    kind: LazyStorageKind
    description: str

@dataclass
class LazyTensor:
    shape: List[int]
    data_type: DataType
    description: str
    
class LazyUnpickler(pickle.Unpickler):
    def __init__(self, fp: IO[bytes], data_base_path: str, zip_file: zipfile.ZipFile):
        super().__init__(fp)
        self.data_base_path = data_base_path
        self.zip_file = zip_file

    def persistent_load(self, pid: Any) -> Any:
        assert pid[0] == 'storage'
        assert isinstance(pid[1], LazyStorageKind)
        data_type = pid[1].data_type
        filename_stem = pid[2]
        filename = self.data_base_path + '/' + filename_stem
        info = self.zip_file.getinfo(filename)
  
        description = f'storage data_type={data_type} path-in-zip={filename} path={self.zip_file.filename}'      
        return LazyStorage(kind=pid[1], description=description)    

    # @staticmethod
    def lazy_rebuild_tensor_v2(storage: Any, storage_offset: Any, size: Any, stride: Any,
                               # pyright: ignore[reportSelfClsParameterName]
                               requires_grad: Any, backward_hooks: Any, metadata: Any = None) -> LazyTensor:
        assert isinstance(storage, LazyStorage)

        description = f'pickled storage_offset={storage_offset} in {storage.description}'
        return LazyTensor(list(size), storage.kind.data_type, description)
    
    CLASSES: Dict[Any, Any] = {
        ('torch._utils', '_rebuild_tensor_v2'):  lazy_rebuild_tensor_v2,
        ('torch', 'BFloat16Storage'): LazyStorageKind(DT_BF16),
        ('torch', 'HalfStorage'): LazyStorageKind(DT_F16),
        ('torch', 'FloatStorage'): LazyStorageKind(DT_F32),
        ('torch', 'IntStorage'): LazyStorageKind(DT_I32)
    }

    def find_class(self, module: str, name: str) -> Any:
        if not module.startswith('torch'):
            return super().find_class(module, name)
        return self.CLASSES[(module, name)]

if __name__ == '__main__':
    path = Path("C:\\llama2\\llama-2-7b-hf\\pytorch_model-00001-of-00002.bin")

    fp = open(path, 'rb')

    zf = zipfile.ZipFile(fp)

    pickle_paths = [name for name in zf.namelist() if name.endswith('.pkl')]
    assert len(pickle_paths) == 1, pickle_paths

    pickle_fp = zf.open(pickle_paths[0], 'r')
    unpickler = LazyUnpickler(pickle_fp,
                              data_base_path=pickle_paths[0][:-4],
                              zip_file=zf)   
    
    model = unpickler.load()
    as_dict = dict(model.items())

    nParams = 0
    nBytes = 0
    count = 0
    for name, lazy_tensor in as_dict.items():
        print(f"{name}: shape={lazy_tensor.shape} type={lazy_tensor.data_type}")
        if len(lazy_tensor.shape) == 1:
            count = lazy_tensor.shape[0]
        else:
            count = lazy_tensor.shape[0] * lazy_tensor.shape[1]

        if lazy_tensor.data_type == DT_F32:
            nBytes += count * 4
        else: #DT_F16
            nBytes += count * 2

        nParams += count

    print(f'参数量={nParams}, 字节数={nBytes}')

model.embed_tokens.weight: shape=[32000, 4096] type=F16
model.layers.0.self_attn.q_proj.weight: shape=[4096, 4096] type=F16
model.layers.0.self_attn.k_proj.weight: shape=[4096, 4096] type=F16
model.layers.0.self_attn.v_proj.weight: shape=[4096, 4096] type=F16
model.layers.0.self_attn.o_proj.weight: shape=[4096, 4096] type=F16
model.layers.0.self_attn.rotary_emb.inv_freq: shape=[64] type=F32
model.layers.0.mlp.gate_proj.weight: shape=[11008, 4096] type=F16
model.layers.0.mlp.up_proj.weight: shape=[11008, 4096] type=F16
model.layers.0.mlp.down_proj.weight: shape=[4096, 11008] type=F16
model.layers.0.input_layernorm.weight: shape=[4096] type=F16
model.layers.0.post_attention_layernorm.weight: shape=[4096] type=F16
model.layers.1.self_attn.q_proj.weight: shape=[4096, 4096] type=F16
model.layers.1.self_attn.k_proj.weight: shape=[4096, 4096] type=F16
model.layers.1.self_attn.v_proj.weight: shape=[4096, 4096] type=F16
model.layers.1.self_attn.o_proj.weight: shape=[4096, 4096] type=F16
model.l

## Transformers读取模型参数

In [7]:
from transformers import AutoModel

# 加载模型
model = AutoModel.from_pretrained("c:/llama2/llama-2-7b-hf")

# 计算模型参数总数
total_params = sum(p.numel() for p in model.parameters())

print(f'Total parameters: {total_params}')

Loading checkpoint shards: 100%|████████████████████████████████████████████████████████| 2/2 [12:03<00:00, 361.70s/it]


Total parameters: 6607343616


## Qwen模型参数

In [1]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained("C:/llama2/Qwen-1_8B", device_map="auto", trust_remote_code=True).eval()

print(model)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

QWenLMHeadModel(
  (transformer): QWenModel(
    (wte): Embedding(151936, 2048)
    (drop): Dropout(p=0.0, inplace=False)
    (rotary_emb): RotaryEmbedding()
    (h): ModuleList(
      (0-23): 24 x QWenBlock(
        (ln_1): RMSNorm()
        (attn): QWenAttention(
          (c_attn): Linear(in_features=2048, out_features=6144, bias=True)
          (c_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (attn_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): RMSNorm()
        (mlp): QWenMLP(
          (w1): Linear(in_features=2048, out_features=5504, bias=False)
          (w2): Linear(in_features=2048, out_features=5504, bias=False)
          (c_proj): Linear(in_features=5504, out_features=2048, bias=False)
        )
      )
    )
    (ln_f): RMSNorm()
  )
  (lm_head): Linear(in_features=2048, out_features=151936, bias=False)
)
